# IndicPulse – Pipeline Exploration Notebook
## Interactive walkthrough of all 5 pipeline steps

Use this notebook to explore results, debug, and iterate on the pipeline.

In [ ]:
# Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s | %(levelname)s | %(message)s')

print('IndicPulse Exploration Notebook ready ✅')

In [ ]:
# ── STEP 1: INGESTION ────────────────────────────────────
from core.ingestion import load_demo_comments, load_csv_comments

# Option A: Demo data
df_raw = load_demo_comments()

# Option B: Your CSV
# df_raw = load_csv_comments('../data/sample_reviews.csv')

# Option C: YouTube (needs API key)
# from core.ingestion import fetch_youtube_comments
# df_raw = fetch_youtube_comments('https://youtu.be/YOUR_VIDEO', api_key='YOUR_KEY')

print(f'Ingested: {len(df_raw)} comments')
df_raw.head()

In [ ]:
# ── STEP 2: PREPROCESSING ────────────────────────────────
from core.preprocessor import preprocess_dataframe, preprocess_comment

# Single comment test
test = preprocess_comment('Baaaahut fire hai yeh video 🔥🔥 ch*tiya mat bolo')
print('Single comment test:')
for k, v in test.items():
    print(f'  {k}: {v}')

print('\nProcessing full dataset...')
df_prep = preprocess_dataframe(df_raw)
print(f'Preprocessed: {len(df_prep)} rows')
print(f'Has profanity: {df_prep["has_profanity"].sum()} comments')
df_prep[['text', 'text_for_mood', 'text_for_topic', 'has_profanity', 'profanity_severity']].head()

In [ ]:
# ── STEP 3: MOOD CLASSIFICATION ──────────────────────────
from core.mood_classifier import MuRILClassifier, classify_dataframe

classifier = MuRILClassifier(device='cpu')  # Change to 'cuda' if GPU available
classifier.load()

df_classified = classify_dataframe(df_prep, classifier=classifier)

print('Mood distribution:')
print(df_classified['mood_label'].value_counts())
df_classified[['text', 'mood_label', 'mood_score', 'is_toxic']].head(10)

In [ ]:
# ── STEP 4: TOPIC CLUSTERING ─────────────────────────────
from core.topic_clusterer import IndicSBERTEmbedder, cluster_topics, add_engagement_weights

df_weighted = add_engagement_weights(df_classified)

# Note: This requires sentence-transformers and bertopic installed
# pip install sentence-transformers bertopic umap-learn hdbscan

try:
    embedder = IndicSBERTEmbedder()
    df_topics, topic_model, topic_summary = cluster_topics(df_weighted, embedder=embedder)
    
    print(f'Topics discovered: {len([k for k in topic_summary if k != -1])}')
    for tid, info in topic_summary.items():
        if tid == -1: continue
        print(f"\n  Cluster: {info['label']} | Size: {info['size']}")
        print(f"  Keywords: {', '.join(info['top_words'][:5])}")
        
except ImportError as e:
    print(f'Topic clustering requires: {e}')
    print('Run: pip install sentence-transformers bertopic umap-learn hdbscan')
    df_topics = df_weighted.copy()
    df_topics['topic_id'] = -1
    df_topics['topic_label'] = 'Unclustered'
    topic_summary = {}

In [ ]:
# ── STEP 5: GEMINI SUMMARY ───────────────────────────────
from core.summarizer import generate_executive_summary
import os

# Set your API key
GEMINI_KEY = os.getenv('GEMINI_API_KEY', '')  # or paste directly

if GEMINI_KEY:
    summary = generate_executive_summary(
        topic_summary=topic_summary,
        df=df_topics,
        api_key=GEMINI_KEY,
        source='demo',
    )
    import json
    print(json.dumps(summary, indent=2))
else:
    print('Set GEMINI_API_KEY to generate the AI executive summary.')
    print('Get your free key at: https://aistudio.google.com/app/apikey')

In [ ]:
# ── One-shot: Full Pipeline ───────────────────────────────
from core.pipeline import run_pipeline

result = run_pipeline(
    source='demo',
    api_key_gemini=os.getenv('GEMINI_API_KEY', ''),
    skip_llm=not bool(os.getenv('GEMINI_API_KEY')),
)

print('Pipeline Result:')
print(f'  Total: {result.meta["total_comments"]}')
print(f'  Constructive: {result.meta["constructive_comments"]}')
print(f'  Toxic: {result.meta["toxic_comments"]}')
print(f'  Topics: {result.meta["topics_found"]}')
print(f'  Pulse Score: {result.pulse_score}/10')
print(f'  Mood: {result.mood_counts}')